# Z-Image Turbo × Fuli — Complete Pipeline Notebook
**AMD OnClick AI / ROCm 6.4 compatible**

A self-contained teaching notebook that walks you from a fresh GPU to a
published artist-identity LoRA in seven sections:

| # | Section | What you do |
|---|---------|-------------|
| 0 | Setup & Downloads | Install packages, log in to HF, download everything |
| 1 | First Impression | Prompt the merged model, get a feel for it |
| 2 | Consistency Problem | See why the same name → different faces |
| 3 | LoRA Training | Train with live loss plot |
| 4 | Scale Explorer | Sweep LoRA scales, pick the best |
| 5 | Compression | Quantise, compare sizes + visual quality |
| 6 | Push to Hub | Publish adapter or merged model (optional) |


---
## 0 · Setup — Install, Auth & Downloads

> Run this block first and only once per environment.
> Everything else pulls from the HF cache — no re-downloads on repeat runs.

In [ ]:
# ── 0A  Install dependencies ────────────────────────────────────────────
import subprocess
import sys
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Torch — uncomment the line that matches your hardware:
pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/rocm6.4')  # AMD ROCm 6.4
# pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/rocm6.2')  # AMD ROCm 6.2
# pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/cu124')    # NVIDIA CUDA 12.4

# diffusers from source (ZImagePipeline not yet on PyPI)
pip('git+https://github.com/huggingface/diffusers')

# core runtime
pip('safetensors>=0.7.0','accelerate>=1.0.0','transformers>=5.0.0',
    'peft>=0.18.0','huggingface_hub>=1.9.0',
    'datasets>=3.0.0','pandas>=2.0.0',
    'matplotlib>=3.9.0','ipywidgets>=8.0.0',
    'optimum-quanto>=0.2.0','Pillow>=10.0.0')

print('All packages installed.')

# support CJK characters in matplotlib
import os
import urllib.request

font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
font_path = "NotoSansCJKsc-Regular.otf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

from matplotlib import font_manager, rcParams
font_manager.fontManager.addfont(font_path)

prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

rcParams["font.family"] = font_name
rcParams["axes.unicode_minus"] = False

In [ ]:
# ── Notebook display helper ──────────────────────────────────────────────
# Saves a high-res file to disk; embeds a small JPEG preview inline.
# Keeps cell outputs compact, avoiding 413 "notebook too large" errors.
import io as _io
import matplotlib.pyplot as _plt
from IPython.display import display as _ipy_display, Image as _IPImage

def show_fig(fig, save_path, save_dpi=130, preview_width=900):
    """Save full-quality image to disk; show a compressed JPEG preview inline."""
    fig.savefig(save_path, bbox_inches='tight', dpi=save_dpi)
    buf = _io.BytesIO()
    fig.savefig(buf, format='jpeg', bbox_inches='tight', dpi=72, quality=82)
    buf.seek(0)
    _plt.close(fig)
    _ipy_display(_IPImage(data=buf.read(), width=preview_width))

print('show_fig helper ready.')


In [ ]:
# ── 0B  HF Auth + Download model + dataset in one shot ──────────────────
import os
from getpass import getpass
from huggingface_hub import login, snapshot_download, hf_hub_download
import torch
from pathlib import Path

HF_TOKEN = getpass('HuggingFace token (hf_...): ')
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN

MODEL_REPO   = 'DownFlow/Z-Image-Turbo-Fuli'
DATASET_REPO = 'DownFlow/fuliji'
PARQUET_FILE = 'dataset.parquet'

print(f'[1/2] Downloading {MODEL_REPO}  (~20 GB, cached on repeat runs) ...')
MODEL_DIR = snapshot_download(MODEL_REPO, token=HF_TOKEN)
print(f'  -> {MODEL_DIR}')

print(f'[2/2] Downloading {DATASET_REPO}/{PARQUET_FILE} ...')
PARQUET_PATH = hf_hub_download(
    repo_id=DATASET_REPO,
    filename=PARQUET_FILE,
    repo_type='dataset',
    token=HF_TOKEN,
)
print(f'  -> {PARQUET_PATH}')

# GPU check
if torch.cuda.is_available():
    d = torch.cuda.get_device_properties(0)
    print(f'GPU: {d.name}  {d.total_memory/2**30:.1f} GB VRAM')
else:
    print('WARNING: no CUDA GPU detected.')
print('Setup complete.')

---
## 1 · First Impression — Prompt the Merged Model

The merged model already knows hundreds of Fuli artists out of the box.
Change `PROMPT` below and run to see what it generates.

In [ ]:
# ── 1A  Load pipeline ────────────────────────────────────────────────────
import torch
from diffusers import ZImagePipeline

print(f'Loading {MODEL_DIR} ...')
pipe = ZImagePipeline.from_pretrained(
    MODEL_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=False,
).to('cuda')
pipe.set_progress_bar_config(disable=False)
print('Pipeline ready.')

In [ ]:
# ── 1B  Prompt  (edit and re-run freely) ─────────────────────────────────
PROMPT   = 'portrait of Liu Yifei, soft studio lighting, high detail, 4k'  # <- edit me
NEG      = 'blurry, low quality, deformed, watermark, text'
STEPS    = 20
CFG      = 4.0
SEED     = 42
RES      = 512
N_IMAGES = 4  # number of samples to generate side-by-side

import matplotlib.pyplot as plt
g = torch.Generator('cuda').manual_seed(SEED)
with torch.no_grad():
    out = pipe(
        prompt=[PROMPT] * N_IMAGES,
        negative_prompt=[NEG] * N_IMAGES,
        height=RES, width=RES,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=g,
    )

fig, axes = plt.subplots(1, N_IMAGES, figsize=(4 * N_IMAGES, 4.2))
if N_IMAGES == 1: axes = [axes]
fig.suptitle(PROMPT, fontsize=11, y=1.01)
for ax, img in zip(axes, out.images):
    ax.imshow(img); ax.axis('off')
plt.tight_layout()
show_fig(fig, '/tmp/section1_gallery.png', save_dpi=120)


---
## 2 · The Character Consistency Problem

The base model generates a *different face* every run even for the same named artist.
The grid below uses 4 fixed seeds — notice the faces drift between columns.
Column 5 shows a real reference photo from the dataset.

This is the core problem LoRA solves.

In [ ]:
# ── 2A  Load dataset, pick two artists ──────────────────────────────────
import pandas as pd, io
from PIL import Image

df = pd.read_parquet(PARQUET_PATH)

def build_caption(artist, ftags, itags):
    ph = ', '.join(str(t) for t in list(ftags)  if t)
    sc = ', '.join(str(t) for t in list(itags)[:8] if t)
    pts = [f'portrait of {artist}'] + ([ph] if ph else []) + ([sc] if sc else [])
    return ', '.join(pts)

top_artists = df['fuliji'].value_counts().head(2).index.tolist()
DEMO_ARTISTS = top_artists
print('Demo artists:', DEMO_ARTISTS)

In [ ]:
# ── 2B  Consistency matrix: same prompt, 4 seeds, + real photo ───────────
import matplotlib.pyplot as plt
import torch

SEEDS  = [0, 7, 42, 99]
STEPS  = 20; CFG = 4.0; RES = 512

fig, axes = plt.subplots(len(DEMO_ARTISTS), len(SEEDS) + 1,
                          figsize=((len(SEEDS)+1)*3.5, len(DEMO_ARTISTS)*3.5))
if len(DEMO_ARTISTS) == 1: axes = [axes]

for row_i, artist in enumerate(DEMO_ARTISTS):
    rows  = df[df['fuliji'] == artist]
    row0  = rows.iloc[0]
    ft = list(row0.get('fuliji_tags') or [])
    it = list(row0.get('image_tags')  or [])
    cap = build_caption(artist, ft, it)

    for col_i, seed in enumerate(SEEDS):
        g = torch.Generator('cuda').manual_seed(seed)
        with torch.no_grad():
            img = pipe(
                prompt=cap,
                negative_prompt='blurry, low quality, deformed, watermark',
                height=RES, width=RES,
                num_inference_steps=STEPS, guidance_scale=CFG, generator=g,
            ).images[0]
        axes[row_i][col_i].imshow(img)
        axes[row_i][col_i].axis('off')
        if row_i == 0:
            axes[row_i][col_i].set_title(f'seed={seed}', fontsize=8)
        if col_i == 0:
            axes[row_i][col_i].set_ylabel(artist, fontsize=9,
                rotation=0, ha='right', va='center', labelpad=70)

    # Real reference photo (last column)
    ref = Image.open(io.BytesIO(row0['image']['bytes'])).convert('RGB')
    s   = min(ref.size)
    ref = ref.crop(((ref.width-s)//2,(ref.height-s)//2,
                    (ref.width+s)//2,(ref.height+s)//2)).resize((RES,RES))
    axes[row_i][-1].imshow(ref); axes[row_i][-1].axis('off')
    if row_i == 0:
        axes[row_i][-1].set_title('Real photo', fontsize=8, color='#2ca02c')

fig.suptitle('Same name, different seeds => different faces  (base model, no LoRA)',
              fontsize=12, y=1.01)
plt.tight_layout()
show_fig(fig, '/tmp/section2_consistency.png', save_dpi=130)


---
## 3 · LoRA Fine-tuning

Train artist-identity adapters with a live loss + LR plot that refreshes
every `PLOT_EVERY` optimiser steps.

> **Quick test:** keep `STEPS = 100` to verify the loop runs, then set
> `STEPS = 2000–5000` for a real adapter.

> **Resume:** set `RESUME_CKPT` to an existing `adapter_checkpoint_XXXXX_ema`
> directory and `RESUME_STEP` to the matching step number.

In [ ]:
# ── 3A  Training configuration  (edit these) ─────────────────────────────
from pathlib import Path

# Data
MIN_ARTIST_IMGS = 5         # skip artists with fewer than this many images
REG_RATIO       = 0.20      # fraction of steps that use regularisation samples

# LoRA architecture
LORA_RANK     = 32
LORA_ALPHA    = 32.0        # lora_alpha / rank = scale; 32/32 = 1.0
TARGET_MODS   = ['to_q','to_k','to_v','w1','w2','w3']

# Optimiser
STEPS         = 100         # <- use 2000-5000 for production
LR            = 1e-4
WARMUP        = 20
WEIGHT_DECAY  = 0.01
GRAD_ACCUM    = 4
BATCH         = 1
EMA_DECAY     = 0.9999
CAP_DROPOUT   = 0.05
TS_BIAS       = 1.2         # timestep sampling bias (>1 = more style steps)

# Resume (leave empty to train from scratch)
RESUME_CKPT   = ''          # path to adapter_checkpoint_XXXXX_ema dir
RESUME_STEP   = 0           # step number that matches the checkpoint

# Output
OUTPUT_DIR    = Path.home() / 'training' / 'lora_fuli_run01'
PLOT_EVERY    = max(1, STEPS // 40)  # refresh plot this often (optimiser steps)
SAVE_EVERY    = max(1, STEPS // 5)   # save checkpoint this often

print('Config ready. Run the next cell to start training.')

In [ ]:
# ── 3B  LoRA training with live loss plot ────────────────────────────────
# All infrastructure is inline; no external script invoked.
import copy, io, math, random, re, time
import numpy as np
import torch
import pandas as pd
from PIL import Image
from pathlib import Path
from torchvision import transforms
from diffusers import AutoencoderKL, FlowMatchEulerDiscreteScheduler, ZImagePipeline
from peft import LoraConfig, get_peft_model
from safetensors.torch import load_file as st_load, save_file as st_save
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import clear_output

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────
VAE_SCALE  = 0.3611; VAE_SHIFT = 0.1159; PMULT = 16

def _round(x, m): return (x // m) * m

def make_tf(side, flip=False):
    to_t = transforms.ToTensor(); norm = transforms.Normalize([0.5],[0.5])
    def tf(img):
        if flip and random.random() < 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = img.convert('RGB'); w, h = img.size
        sc = side / min(w, h)
        nw, nh = max(side, int(w*sc)), max(side, int(h*sc))
        img = img.resize((nw, nh), Image.LANCZOS)
        cw, ch = _round(nw, PMULT), _round(nh, PMULT)
        img = img.crop(((nw-cw)//2,(nh-ch)//2,(nw+cw)//2,(nh+ch)//2))
        return norm(to_t(img))
    return tf

def enc_img(vae, pv):
    with torch.no_grad(): lat = vae.encode(pv).latent_dist.sample()
    return (lat - VAE_SHIFT) * VAE_SCALE

def enc_txt(p, prompts, device, dtype):
    with torch.no_grad():
        r = p.encode_prompt(prompts[0] if len(prompts)==1 else prompts,
                            device=device, do_classifier_free_guidance=False)
    return [f.to(dtype=dtype) for f in r[0]]

def get_noisy(sched, lat, noise, t_idx):
    sig = sched.sigmas.to(lat.device, lat.dtype)[t_idx].view(-1,1,1,1)
    return (1-sig)*lat + sig*noise

def fm_loss(pred, noise, lat, sigmas, gamma=5.0):
    target = noise - lat
    diff = (pred.float() - target.float())**2
    ps   = diff.mean(dim=list(range(1, diff.ndim)))
    snr  = (1-sigmas)**2 / (sigmas**2 + 1e-8)
    return (snr.clamp(max=gamma) * ps).mean()

def lr_fn(step):
    if step < WARMUP: return (step+1)/max(WARMUP,1)
    p = (step-WARMUP)/max(STEPS-WARMUP,1)
    return 0.1 + 0.9*0.5*(1+math.cos(math.pi*p))

# ── Dataset ───────────────────────────────────────────────────────────────
print('Loading dataset ...')
df_tr = pd.read_parquet(PARQUET_PATH)
if MIN_ARTIST_IMGS > 1:
    cnt = df_tr['fuliji'].value_counts()
    df_tr = df_tr[df_tr['fuliji'].isin(cnt[cnt >= MIN_ARTIST_IMGS].index)]
    print(f'  {df_tr["fuliji"].nunique()} artists, {len(df_tr)} images')

tf = make_tf(512, flip=True)
records = []
for _, row in df_tr.iterrows():
    ft = list(row.get('fuliji_tags') or [])
    it = list(row.get('image_tags')  or [])
    records.append((row['image']['bytes'],
                    build_caption(str(row['fuliji']), ft, it)))
print(f'  {len(records)} training records ready.')

def art_iter(bs=1):
    idx = list(range(len(records)))
    while True:
        random.shuffle(idx)
        bimgs, btxt = [], []
        for i in idx:
            raw, cap = records[i]
            try:
                img = Image.open(io.BytesIO(raw)).convert('RGB')
                bimgs.append(tf(img)); btxt.append(cap)
            except: continue
            if len(bimgs) == bs:
                yield torch.stack(bimgs), btxt; bimgs, btxt = [], []

# ── Pipeline ──────────────────────────────────────────────────────────────
device = torch.device('cuda'); dtype = torch.bfloat16
print(f'Loading pipeline from {MODEL_DIR} ...')
_p = ZImagePipeline.from_pretrained(
    MODEL_DIR, torch_dtype=dtype, low_cpu_mem_usage=False).to(device)
_p.vae.requires_grad_(False).eval()
for _n in ('text_encoder','tokenizer','cap_embedder'):
    _m = getattr(_p, _n, None)
    if _m is not None and hasattr(_m, 'requires_grad_'):
        _m.requires_grad_(False)
        if hasattr(_m, 'eval'): _m.eval()
sched = _p.scheduler; sched.set_timesteps(1000)

# ── LoRA ──────────────────────────────────────────────────────────────────
print('Injecting LoRA ...')
_p.transformer.requires_grad_(False)
lcfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                  target_modules=TARGET_MODS, bias='none')
peft_tr = get_peft_model(_p.transformer, lcfg)
peft_tr.print_trainable_parameters()
_p.transformer = peft_tr

if RESUME_CKPT:
    print(f'Resuming from {RESUME_CKPT} ...')
    _ckpt = Path(RESUME_CKPT) / 'adapter_model.safetensors'
    _st   = st_load(str(_ckpt), device='cpu')
    _rm   = {re.sub(r'\.(lora_A|lora_B|lora_embedding_A|lora_embedding_B)\.weight$',
                    r'.\1.default.weight', k): v for k,v in _st.items()}
    peft_tr.load_state_dict(_rm, strict=False)
    print(f'  Loaded {len(_rm)} tensors.')

# ── EMA ───────────────────────────────────────────────────────────────────
ema_st = {k: v.clone().detach() for k,v in peft_tr.state_dict().items()
          if 'lora_' in k}

# ── Optimiser ─────────────────────────────────────────────────────────────
lp  = [p for p in peft_tr.parameters() if p.requires_grad]
opt = torch.optim.AdamW(lp, lr=LR, betas=(0.9,0.999), eps=1e-8,
                        weight_decay=WEIGHT_DECAY)
sched_lr = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

# ── Training loop ─────────────────────────────────────────────────────────
peft_tr.train()
a_iter = art_iter(BATCH)
print(f'Starting training: {STEPS} steps, lr={LR:.1e}, grad_accum={GRAD_ACCUM}')

step_log, loss_log, lr_log = [], [], []
global_step = RESUME_STEP
accum_loss  = 0.0; accum_n = 0
t0 = time.time(); opt.zero_grad()

for micro in range(STEPS * GRAD_ACCUM):
    if global_step >= STEPS: break
    if micro < RESUME_STEP * GRAD_ACCUM: continue

    pv, txts = next(a_iter)
    if torch.rand(1).item() < CAP_DROPOUT: txts = [''] * len(txts)
    pv = pv.to(device=device, dtype=dtype)

    lat = enc_img(_p.vae, pv)
    cap = enc_txt(_p, txts, device, dtype)

    B  = lat.shape[0]
    u  = torch.sigmoid(torch.randn(B, device=device) * TS_BIAS)
    ti = (u * sched.config.num_train_timesteps).long().clamp(
             0, sched.config.num_train_timesteps - 1)
    sig = sched.sigmas.to(device=device, dtype=dtype)[ti]
    nz  = torch.randn_like(lat)
    nlt = get_noisy(sched, lat, nz, ti)
    t_ts = sched.timesteps.to(device)[ti]

    pred = peft_tr(
        hidden_states=nlt, timestep=t_ts,
        encoder_hidden_states=cap[0] if isinstance(cap, list) else cap,
        return_dict=False)[0]

    loss = fm_loss(pred, nz, lat, sig) / GRAD_ACCUM
    loss.backward()
    accum_loss += loss.item() * GRAD_ACCUM; accum_n += 1

    if (micro + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(lp, 1.0)
        opt.step(); sched_lr.step(); opt.zero_grad()
        global_step += 1

        cur_loss = accum_loss / accum_n
        cur_lr   = opt.param_groups[0]['lr']
        step_log.append(global_step)
        loss_log.append(cur_loss)
        lr_log.append(cur_lr)
        accum_loss = 0.0; accum_n = 0

        # EMA update
        if EMA_DECAY > 0:
            with torch.no_grad():
                for k, v in peft_tr.state_dict().items():
                    if k in ema_st:
                        ema_st[k].mul_(EMA_DECAY).add_(v.detach().cpu(),
                                                        alpha=1-EMA_DECAY)

        # Live plot — saved to disk; compact JPEG shown inline
        if global_step % PLOT_EVERY == 0 or global_step == STEPS:
            elapsed = time.time() - t0
            spd = global_step / max(elapsed, 1e-6)
            eta = (STEPS - global_step) / spd
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5))
            ax1.plot(step_log, loss_log, color='#4c90f0', linewidth=1.5, alpha=0.6)
            if len(loss_log) > 8:
                w  = max(3, len(loss_log)//10)
                sm = np.convolve(loss_log, np.ones(w)/w, mode='valid')
                ax1.plot(step_log[w-1:], sm, color='#e07020', linewidth=2, label=f'EMA-{w}')
                ax1.legend(fontsize=8)
            ax1.set(xlabel='Step', ylabel='Loss', title='Training Loss')
            ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
            ax2.semilogy(step_log, lr_log, color='#2ab77c', linewidth=1.8)
            ax2.set(xlabel='Step', ylabel='LR (log)', title='Learning Rate')
            fig.text(0.5, 1.01,
                f'Step {global_step}/{STEPS}  loss={cur_loss:.4f}  '
                f'lr={cur_lr:.2e}  {spd:.1f} it/s  ETA {eta/60:.1f} min',
                ha='center', fontsize=10)
            plt.tight_layout()
            show_fig(fig, str(OUTPUT_DIR / 'live_loss.png'), save_dpi=100)

        # Checkpoint
        if global_step % SAVE_EVERY == 0 or global_step == STEPS:
            ck = OUTPUT_DIR / f'adapter_checkpoint_{global_step:05d}'
            peft_tr.save_pretrained(str(ck))
            if EMA_DECAY > 0:
                ek = OUTPUT_DIR / f'adapter_checkpoint_{global_step:05d}_ema'
                ek.mkdir(parents=True, exist_ok=True)
                st_save(ema_st, str(ek / 'adapter_model.safetensors'))
            print(f'  Checkpoint saved at step {global_step}')

ADAPTER_PATH = OUTPUT_DIR / 'final_adapter'
peft_tr.save_pretrained(str(ADAPTER_PATH))
print(f'Training complete. Adapter -> {ADAPTER_PATH}')


In [ ]:
# ── 3C  Final loss summary ────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5))
ax1.plot(step_log, loss_log, alpha=0.4, color='#4c90f0', linewidth=1, label='raw')
if len(loss_log) > 8:
    w  = max(3, len(loss_log) // 8)
    sm = np.convolve(loss_log, np.ones(w)/w, mode='valid')
    ax1.plot(step_log[w-1:], sm, color='#e07020', linewidth=2.5, label=f'EMA-{w}')
ax1.legend(fontsize=9)
ax1.set(xlabel='Step', ylabel='Loss', title='LoRA Training Loss (final summary)')
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax2.semilogy(step_log, lr_log, color='#2ab77c', linewidth=2)
ax2.set(xlabel='Step', ylabel='LR (log scale)', title='Learning Rate (cosine decay)')
plt.tight_layout()
show_fig(fig, str(OUTPUT_DIR / 'final_loss.png'), save_dpi=140)
print(f'Final loss: {loss_log[-1]:.4f}  (start: {loss_log[0]:.4f})')


---
## 4 · Scaling Factor Explorer

The LoRA scale multiplier controls how strongly the adapter steers the model.

| Scale | Effect |
|-------|--------|
| < 0.8 | Softer style influence, more base-model freedom |
| 1.0   | Training default (alpha/rank = 1) |
| 1.2 – 1.5 | Stronger identity, slight risk of over-sharpening |
| > 1.8 | Heavy style lock-in, may distort anatomy |

> **Single mode:** set `SCALE_MODE = 'single'` and `SCALE_VALUE` to test one number.  
> **Sweep mode:** set `SCALE_MODE = 'sweep'` and `SCALE_RANGE` to a list — a comparison
> panel is generated and saved inline. Panels persist in the notebook output.

In [ ]:
# ── 4A  Scale config  (edit and re-run freely) ──────────────────────────
SCALE_MODE   = 'sweep'          # 'single' or 'sweep'
SCALE_VALUE  = 1.0              # used when SCALE_MODE == 'single'
SCALE_RANGE  = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]   # used when SCALE_MODE == 'sweep'

SCALE_ARTISTS = DEMO_ARTISTS    # override to test specific artists
SCALE_SEEDS   = [42, 7]         # seeds per artist
SCALE_STEPS   = 20
SCALE_CFG     = 3.5
SCALE_RES     = 512

print('Config set. Run the next cell to generate panels.')

In [ ]:
# ── 4B  Generate and display scale comparison panels ─────────────────────
import io as _io, gc, torch
from PIL import Image, ImageDraw, ImageFont
from diffusers import ZImagePipeline
from peft import PeftModel
from IPython.display import display as _ipy_display, Image as _IPImage

scales = SCALE_RANGE if SCALE_MODE == 'sweep' else [SCALE_VALUE]

def sq(img, s):
    w, h = img.size; m = min(w, h)
    return img.crop(((w-m)//2,(h-m)//2,(w+m)//2,(h+m)//2)).resize((s,s),Image.LANCZOS)

def lbl_bar(w, h, txt, bg=(40,40,70), fg=(230,230,230), fs=13):
    bar = Image.new('RGB', (w, h), bg); d = ImageDraw.Draw(bar)
    try:  fnt = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', fs)
    except: fnt = ImageFont.load_default()
    d.text((5, (h-fs)//2), txt, fill=fg, font=fnt); return bar

CELL_S = SCALE_RES; SEP = 4; LBAR = 28
panels_saved = []

for artist in SCALE_ARTISTS:
    rows  = df[df['fuliji'] == artist]
    row0  = rows.iloc[0]
    ft = list(row0.get('fuliji_tags') or [])
    it = list(row0.get('image_tags')  or [])
    cap = build_caption(artist, ft, it)

    ref_img = sq(Image.open(_io.BytesIO(row0['image']['bytes'])).convert('RGB'), CELL_S)

    # Canvas:  (1 ref col + N scale cols) x N_seeds rows
    ncols = 1 + len(scales)
    nrows = len(SCALE_SEEDS)
    W = ncols * CELL_S + (ncols-1) * SEP
    H = LBAR*2 + nrows * CELL_S + (nrows-1) * SEP
    canvas = Image.new('RGB', (W, H), (10,10,10))

    # Title bar
    canvas.paste(lbl_bar(W, LBAR, f'{artist}  |  {cap[:70]}', bg=(30,30,60)), (0,0))

    # Column headers
    canvas.paste(lbl_bar(CELL_S, LBAR, 'Real photo', bg=(15,60,15)), (0, LBAR))
    for ci, sc in enumerate(scales):
        x = (ci+1)*(CELL_S+SEP)
        canvas.paste(lbl_bar(CELL_S, LBAR, f'scale={sc}', bg=(40,40,80)), (x, LBAR))

    for ri, seed in enumerate(SCALE_SEEDS):
        y = LBAR*2 + ri*(CELL_S+SEP)
        canvas.paste(ref_img, (0, y))

        for ci, sc in enumerate(scales):
            # fresh pipe with this scale
            lp2 = ZImagePipeline.from_pretrained(
                MODEL_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=False).to('cuda')
            lp2.transformer = PeftModel.from_pretrained(
                lp2.transformer, str(ADAPTER_PATH), is_trainable=False)
            if sc != 1.0:
                for mod in lp2.transformer.modules():
                    if hasattr(mod, 'scaling'):
                        mod.scaling = {k: v*sc for k,v in mod.scaling.items()}
            lp2.set_progress_bar_config(disable=True)

            g = torch.Generator('cuda').manual_seed(seed)
            with torch.no_grad():
                img = lp2(prompt=cap, height=SCALE_RES, width=SCALE_RES,
                          num_inference_steps=SCALE_STEPS, guidance_scale=SCALE_CFG,
                          generator=g).images[0]
            del lp2; gc.collect(); torch.cuda.empty_cache()
            canvas.paste(sq(img, CELL_S), ((ci+1)*(CELL_S+SEP), y))

    safe  = artist.replace('/','_')
    out_p = OUTPUT_DIR / f'scale_sweep_{safe}.png'
    canvas.save(str(out_p))
    panels_saved.append(out_p)
    print(f'  Saved: {out_p}')

# Display compact JPEG thumbnails inline (full-res on disk)
PREVIEW_W = 900
for pp in panels_saved:
    pil = Image.open(str(pp))
    ph  = int(pil.height * PREVIEW_W / pil.width)
    thumb = pil.resize((PREVIEW_W, ph), Image.LANCZOS)
    buf = _io.BytesIO()
    thumb.save(buf, format='JPEG', quality=80)
    buf.seek(0)
    print(pp.name)
    _ipy_display(_IPImage(data=buf.read()))
print('All panels displayed and saved.')


---
## 5 · Compression & Quantisation

Reduce disk size and VRAM usage. All formats saved as **safetensors**.

| Method | Size | Speed | Visual quality |
|--------|------|-------|----------------|
| `bfloat16` | ~21 GB | fastest | reference |
| `float8` | ~10.5 GB | fast | ~same |
| `int8` | ~10.5 GB | fast | slight softness |
| `int4` | ~5.3 GB | slower | noticeable softness |

> Uses `optimum-quanto` (pure PyTorch, works on AMD ROCm).

In [ ]:
# ── 5A  Quant config ─────────────────────────────────────────────────────
QUANT_METHODS  = ['bfloat16', 'float8', 'int8', 'int4']
QUANT_DIR      = OUTPUT_DIR / 'quant_models'
QUANT_DIR.mkdir(parents=True, exist_ok=True)

QUANT_PROMPT = f'portrait of {DEMO_ARTISTS[0]}, soft studio lighting, 4k'
QUANT_SEED   = 42
QUANT_STEPS  = 20
QUANT_CFG    = 3.5
QUANT_RES    = 512

print('Config set. Run the next cell to quantise and compare.')

In [ ]:
# ── 5B  Quantise, save, compare ──────────────────────────────────────────
import gc, torch
from diffusers import ZImagePipeline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    from optimum.quanto import quantize, freeze, qfloat8, qint8, qint4
    QUANTO = True
except ImportError:
    QUANTO = False
    print('optimum-quanto unavailable — only bfloat16 will run.')

_QMAP = {
    'bfloat16': None,
    'float8':   qfloat8 if QUANTO else None,
    'int8':     qint8   if QUANTO else None,
    'int4':     qint4   if QUANTO else None,
}

results = {}  # method -> {size_gb, image}

for method in QUANT_METHODS:
    qtype = _QMAP.get(method)
    if qtype is None and method != 'bfloat16':
        print(f'[SKIP] {method}'); continue
    print(f'\nQuantising: {method} ...')

    p = ZImagePipeline.from_pretrained(
        MODEL_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=False).to('cuda')
    if qtype is not None:
        quantize(p.transformer, weights=qtype)
        freeze(p.transformer)

    p.set_progress_bar_config(disable=True)
    g = torch.Generator('cuda').manual_seed(QUANT_SEED)
    with torch.no_grad():
        img = p(prompt=QUANT_PROMPT, height=QUANT_RES, width=QUANT_RES,
                num_inference_steps=QUANT_STEPS, guidance_scale=QUANT_CFG,
                generator=g).images[0]

    out_d = QUANT_DIR / method
    p.save_pretrained(str(out_d))

    total = sum(f.stat().st_size for f in out_d.rglob('*.safetensors'))
    gb    = total / 2**30
    print(f'  Size: {gb:.2f} GB  -> {out_d}')
    results[method] = {'size_gb': gb, 'image': img}
    del p; gc.collect(); torch.cuda.empty_cache()

# ── Visual comparison ─────────────────────────────────────────────────────
n = len(results)
fig = plt.figure(figsize=(4*n + 2, 7))
gs  = gridspec.GridSpec(2, n, height_ratios=[5, 1.2], hspace=0.35)
colours = ['#4c90f0','#e07020','#2ab77c','#c04040','#9050c0']

for ci, (method, info) in enumerate(results.items()):
    ax = fig.add_subplot(gs[0, ci])
    ax.imshow(info['image']); ax.axis('off')
    ax.set_title(f'{method}\n{info["size_gb"]:.2f} GB', fontsize=10)

ax_bar = fig.add_subplot(gs[1, :])
names  = list(results.keys())
sizes  = [v['size_gb'] for v in results.values()]
bars   = ax_bar.bar(names, sizes, color=colours[:n])
ax_bar.set(ylabel='Disk size (GB)', title='Safetensors file size by quant method')
for bar, sz in zip(bars, sizes):
    ax_bar.text(bar.get_x() + bar.get_width()/2, sz + 0.05,
                f'{sz:.2f}', ha='center', fontsize=9)

show_fig(fig, str(QUANT_DIR / 'quant_comparison.png'), save_dpi=140)
print(f'Comparison saved to {QUANT_DIR}/quant_comparison.png')


In [ ]:
# ── 5C  Pick the winner ───────────────────────────────────────────────────
BEST_QUANT    = 'int8'      # <- edit to whichever method looked best
BEST_MODEL_DIR = QUANT_DIR / BEST_QUANT

print(f'Best model: {BEST_MODEL_DIR}')
if BEST_QUANT in results:
    print(f'  Size: {results[BEST_QUANT]["size_gb"]:.2f} GB')
print('Proceed to Section 6 to push, or use BEST_MODEL_DIR for local inference.')

---
## 6 · Push to Hub *(optional)*

Upload the compressed merged model, the LoRA adapter, or both.
The repos are created automatically if they don't exist.

In [ ]:
# ── 6A  Upload config ────────────────────────────────────────────────────
PUSH_MERGED  = True
PUSH_ADAPTER = True

MERGED_REPO  = 'your-org/Z-Image-Turbo-Fuli'      # <- your HF repo
ADAPTER_REPO = 'your-org/Z-Image-Turbo-Fuli-LoRA' # <- your HF LoRA repo
PRIVATE      = True

print('Config set. Run the next cell to upload.')

In [ ]:
# ── 6B  Upload ───────────────────────────────────────────────────────────
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

if PUSH_MERGED:
    print(f'Creating repo {MERGED_REPO} ...')
    api.create_repo(repo_id=MERGED_REPO, repo_type='model',
                    private=PRIVATE, exist_ok=True)
    print(f'Uploading {BEST_MODEL_DIR} ...')
    api.upload_folder(folder_path=str(BEST_MODEL_DIR), repo_id=MERGED_REPO,
                      repo_type='model',
                      commit_message=f'upload: {BEST_QUANT} quantised merged model')
    print(f'  https://huggingface.co/{MERGED_REPO}')

if PUSH_ADAPTER:
    print(f'Creating repo {ADAPTER_REPO} ...')
    api.create_repo(repo_id=ADAPTER_REPO, repo_type='model',
                    private=PRIVATE, exist_ok=True)
    print(f'Uploading {ADAPTER_PATH} ...')
    api.upload_folder(folder_path=str(ADAPTER_PATH), repo_id=ADAPTER_REPO,
                      repo_type='model',
                      commit_message='upload: LoRA adapter (PEFT safetensors)')
    print(f'  https://huggingface.co/{ADAPTER_REPO}')

print('Upload complete.')

---
## Done!

You have:
1. Explored the pre-trained merged model with free prompts
2. Seen the character consistency problem clearly
3. Trained a LoRA adapter with a live loss plot
4. Found the best LoRA scale for your use case
5. Quantised and benchmarked the model (safetensors throughout)
6. *(optionally)* Published your work to Hugging Face

**Tips for better results:**
- Raise `STEPS` to 2000–5000 for production quality
- Try `LORA_RANK` values 16, 32, 64 to trade capacity vs. VRAM
- Adjust `REG_RATIO` if the model forgets non-Fuli prompts
- Use `SCALE_MODE = 'sweep'` after each run to find the sweet spot